# Gold Standard Data Generator

Appends new, fully normally distributed years to `mood-gold-standard.csv`.

Set `yearsToAdd` (and optionally `startDateWhenEmpty`) below and run all cells. If the CSV contains no data rows yet, generation starts at `startDateWhenEmpty`; otherwise it continues seamlessly after the last existing date.

For each column a value is drawn from a normal distribution centred in the middle of its allowed range, then clipped to the range and rounded to an integer (a discretised, truncated normal). The ranges come from the PHQ-9 questionnaire: Mood 1–10, PHQ9_1/2/4/5: 0–3, PHQ9_3/6/7: 0–2, PHQ9_8/9: 0–1.

In [137]:
// --- Configuration ---
val yearsToAdd = 2        // how many full calendar years to append (or generate from scratch)
val randomSeed = 42L      // fixed seed for reproducible output; change for different data
val csvPath = "./mood-gold-standard.csv"
// Used only when the CSV is empty (no data rows yet). Format: yyyy-mm-dd.
val startDateWhenEmpty = "2026-01-01"

In [138]:
import java.io.File
import java.time.LocalDate
import java.util.Locale
import kotlin.random.Random

// Allowed inclusive integer range per column.
data class Column(val min: Int, val max: Int)

val moodColumn = Column(1, 10)
val phqColumns = listOf(
    Column(0, 3), // PHQ9_1
    Column(0, 3), // PHQ9_2
    Column(0, 2), // PHQ9_3
    Column(0, 3), // PHQ9_4
    Column(0, 3), // PHQ9_5
    Column(0, 2), // PHQ9_6
    Column(0, 2), // PHQ9_7
    Column(0, 1), // PHQ9_8
    Column(0, 1)  // PHQ9_9
)

val random = Random(randomSeed)

// Draw one integer from a normal distribution centred in the range, clipped and rounded.
// mean = range midpoint, sd = quarter of the range width so ~95% of mass lands inside before clipping.
fun sample(column: Column): Int {
    val mean = (column.min + column.max) / 2.0
    val sd = (column.max - column.min) / 4.0
    // Box-Muller transform for a standard normal variate.
    val u1 = random.nextDouble().coerceAtLeast(1e-12)
    val u2 = random.nextDouble()
    val z = Math.sqrt(-2.0 * Math.log(u1)) * Math.cos(2.0 * Math.PI * u2)
    val value = Math.round(mean + z * sd).toInt()
    return value.coerceIn(column.min, column.max)
}

In [140]:
val file = File(csvPath)
require(file.exists()) { "CSV not found at $csvPath" }

val allLines = file.readLines().filter { it.isNotBlank() }
val dateStart = Regex("^\\d{4}-\\d{2}-\\d{2}")
val dataLines = allLines.filter { dateStart.containsMatchIn(it.trim()) }

var firstNewDate: LocalDate
var lastDate: LocalDate?

if (dataLines.isEmpty()) {
    lastDate = null
    firstNewDate = LocalDate.parse(startDateWhenEmpty)
    println("CSV has no data rows — generating from scratch, starting $firstNewDate")
} else {
    lastDate = LocalDate.parse(dataLines.last().substringBefore(",").trim())
    firstNewDate = (lastDate?.plusDays(1) ?: println("Last existing date: $lastDate")) as LocalDate
}

val endExclusive = LocalDate.of(firstNewDate.year + yearsToAdd, 1, 1)

val newRows = StringBuilder()
var day = firstNewDate
var count = 0
while (day.isBefore(endExclusive)) {
    val mood = sample(moodColumn)
    val phq = phqColumns.joinToString(",") { sample(it).toString() }
    newRows.append("$day, $mood,$phq\n")
    day = day.plusDays(1)
    count++
}

println("Generating $count new days: $firstNewDate .. ${endExclusive.minusDays(1)}")
println("Preview (first 3 new rows):")
newRows.toString().lines().filter { it.isNotBlank() }.take(3).forEach { println("  $it") }

CSV has no data rows — generating from scratch, starting 2026-01-01
Generating 730 new days: 2026-01-01 .. 2027-12-31
Preview (first 3 new rows):
  2026-01-01, 9,3,0,2,2,2,1,0,0,0
  2026-01-02, 3,1,2,2,2,2,1,1,0,0
  2026-01-03, 6,1,3,1,2,1,1,1,0,1


## Write to File

The cell below actually appends the generated rows to the CSV. Review the preview above first; running this modifies `mood-gold-standard.csv` in place.

In [141]:
// Ensure the file ends with a newline before appending, then write the new rows.
val needsNewline = file.readText().let { it.isNotEmpty() && !it.endsWith("\n") }
file.appendText((if (needsNewline) "\n" else "") + newRows.toString())
println("Appended $count rows to $csvPath")
println("File now has ${file.readLines().filter { it.isNotBlank() }.size} lines (including header).")

Appended 730 rows to ./mood-gold-standard.csv
File now has 731 lines (including header).
